In [1]:
pip install pandas numpy matplotlib scikit-learn

In [4]:
pip install statsmodels prophet xgboost tensorflow

  Using cached prophet-1.3.0-py3-none-win_amd64.whl.metadata (3.6 kB)
  Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl.metadata (4.5 kB)
  Using cached cmdstanpy-1.3.0-py3-none-any.whl.metadata (4.2 kB)
Using cached prophet-1.3.0-py3-none-win_amd64.whl (12.1 MB)
Using cached tensorflow-2.21.0-cp312-cp312-win_amd64.whl (350.9 MB)
Using cached cmdstanpy-1.3.0-py3-none-any.whl (99 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not install packages due to an OSError: [WinError 32] The process cannot access the file because it is being used by another process: 'C:\\Users\\prana\\anaconda3\\Lib\\site-packages\\cmdstanpy\\compilation.py'
Consider using the `--user` option or check the permissions.



In [10]:
pip install prophet

Note: you may need to restart the kernel to use updated packages.


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error

from statsmodels.tsa.statespace.sarimax import SARIMAX

from prophet import Prophet

from xgboost import XGBRegressor

from sklearn.preprocessing import MinMaxScaler

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

import holidays

In [6]:
pip install fastapi uvicorn openpyxl holidays

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   ---------------------------------------- 2.1/2.1 MB 9.6 MB/s eta 0:00:00
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.20.1
    Uninstalling pydantic_core-2.20.1:
      Successfully uninstalled pydantic_core-2.20.1
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.8.2
    Uninstalling pydantic-2.8.2:
      Successfully uninstalled pydantic-2.8.2
Note: you may need to restart the kernel to use updated packages.


In [18]:
df = pd.read_csv("Forecasting.csv")

In [20]:
print(df.head())

        State       Date          Total   Category
0     Alabama  1/12/2019   109,574,036   Beverages
1     Arizona  1/12/2019   109,101,595   Beverages
2    Arkansas  1/12/2019    58,049,432   Beverages
3  California  1/12/2019   444,766,891   Beverages
4    Colorado  1/12/2019    89,816,716   Beverages


In [22]:
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8084 entries, 0 to 8083
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   State     8084 non-null   object
 1   Date      8084 non-null   object
 2   Total     8084 non-null   object
 3   Category  8084 non-null   object
dtypes: object(4)
memory usage: 252.8+ KB
None


In [32]:
df['Date'] = pd.to_datetime(df['Date'], format='mixed', dayfirst=True)

In [34]:
df = df.sort_values(['State', 'Date'])

In [36]:
all_dates = pd.date_range(
    start=df['Date'].min(),
    end=df['Date'].max(),
    freq='D'
)

In [40]:
print(df.columns)

Index(['State', 'Date', 'Total', 'Category'], dtype='object')


Reindex Data

In [52]:
final_df = pd.DataFrame()

states = df['State'].unique()

for state in states:

    temp = df[df['State'] == state].copy()

    temp = temp.set_index('Date')

    all_dates = pd.date_range(start=temp.index.min(),
                              end=temp.index.max(),
                              freq='D')

    temp = temp.reindex(all_dates)

    temp['State'] = state

    temp['Total'] = temp['Total'].astype(str).str.replace(',', '')
    temp['Total'] = pd.to_numeric(temp['Total'], errors='coerce')

    temp['Total'] = temp['Total'].interpolate()

    temp['Category'] = temp['Category'].ffill()

    temp = temp.reset_index()

    temp.rename(columns={'index': 'Date'}, inplace=True)

    final_df = pd.concat([final_df, temp], ignore_index=True)

Feature Engineering - Lag Features

In [56]:
final_df['lag_1'] = final_df['Total'].shift(1)

final_df['lag_7'] = final_df['Total'].shift(7)

final_df['lag_30'] = final_df['Total'].shift(30)

Rolling Features

In [58]:
final_df['rolling_mean_7'] = final_df['Total'].rolling(7).mean()

final_df['rolling_std_7'] = final_df['Total'].rolling(7).std()

Date Features

In [60]:
final_df['day_of_week'] = final_df['Date'].dt.dayofweek

final_df['month'] = final_df['Date'].dt.month

final_df['week'] = final_df['Date'].dt.isocalendar().week

In [62]:
india_holidays = holidays.India()

final_df['holiday_flag'] = final_df['Date'].isin(india_holidays).astype(int)

Remove Null Values

In [64]:
final_df = final_df.dropna()

In [66]:
train_size = int(len(final_df) * 0.8)

train = final_df.iloc[:train_size]

test = final_df.iloc[train_size:]

Sarima Model

In [75]:
# Create copy
train = train.copy()

# Convert index to datetime
train.index = pd.to_datetime(train.index)

# Remove duplicate dates
train = train.groupby(train.index)['Total'].sum()

# Convert back to dataframe
train = train.to_frame()

# Set daily frequency
train = train.asfreq('D')

# Fill missing values
train['Total'] = train['Total'].interpolate()

# SARIMA Model
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima_model = SARIMAX(
    train['Total'],
    order=(1,1,1),
    seasonal_order=(1,1,1,7)
)

sarima_fit = sarima_model.fit()

# Forecast
sarima_pred = sarima_fit.forecast(len(test))

print(sarima_pred)

C:\Users\prana\anaconda3\Lib\site-packages\statsmodels\base\model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


2023-05-08    7.722266e+09
2023-05-09    7.745809e+09
2023-05-10    7.768353e+09
2023-05-11    7.790248e+09
2023-05-12    7.811197e+09
                  ...     
2054-03-01    5.428628e+10
2054-03-02    5.428888e+10
2054-03-03    5.429175e+10
2054-03-04    5.429355e+10
2054-03-05    5.429452e+10
Freq: D, Name: predicted_mean, Length: 11260, dtype: float64


Prophet Model

In [82]:
from prophet import Prophet
from sklearn.metrics import mean_absolute_error

# Prepare train data
prophet_train = train.reset_index()[['Date', 'Total']]

# Rename columns for Prophet
prophet_train.columns = ['ds', 'y']

# Create model
prophet_model = Prophet()

# Train model
prophet_model.fit(prophet_train)

# Create future dates
future = prophet_model.make_future_dataframe(
    periods=len(test),
    freq='D'
)

# Predict
forecast = prophet_model.predict(future)

# Get predictions for test period
prophet_pred = forecast['yhat'].tail(len(test)).values

# Evaluate model
prophet_mae = mean_absolute_error(
    test['Total'],
    prophet_pred
)

print("Prophet MAE:", prophet_mae)

12:44:53 - cmdstanpy - INFO - Chain [1] start processing
12:44:55 - cmdstanpy - INFO - Chain [1] done processing


Prophet MAE: 9636909836.567816


XGBoost

In [95]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error

# Create required features
train['lag_1'] = train['Total'].shift(1)
train['lag_7'] = train['Total'].shift(7)
train['lag_30'] = train['Total'].shift(30)

train['rolling_mean_7'] = train['Total'].rolling(7).mean()
train['rolling_std_7'] = train['Total'].rolling(7).std()

train['day_of_week'] = train.index.dayofweek
train['month'] = train.index.month

train['holiday_flag'] = 0

# Remove missing values
train = train.dropna()

features = [
    'lag_1',
    'lag_7',
    'lag_30',
    'rolling_mean_7',
    'rolling_std_7',
    'day_of_week',
    'month',
    'holiday_flag'
]
split_index = int(len(train) * 0.8)

train_data = train.iloc[:split_index]
test_data = train.iloc[split_index:]

# X and y
X_train = train_data[features]
y_train = train_data['Total']

X_test = test_data[features]
y_test = test_data['Total']

xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_test)

xgb_mae = mean_absolute_error(y_test, xgb_pred)

print("XGBoost MAE:", xgb_mae)

XGBoost MAE: 34421288.76171876


In [99]:
from sklearn.preprocessing import MinMaxScaler

# Create scaler
scaler = MinMaxScaler()

# Scale Total column
scaled_sales = scaler.fit_transform(
    final_df[['Total']]
)

In [102]:
import numpy as np

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

scaler = MinMaxScaler()

scaled_sales = scaler.fit_transform(
    final_df[['Total']]
)

def create_sequences(data, seq_length):

    X = []
    y = []

    for i in range(seq_length, len(data)):

        X.append(data[i-seq_length:i])

        y.append(data[i])

    return np.array(X), np.array(y)

X, y = create_sequences(scaled_sales, 30)

split = int(len(X) * 0.8)

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

# Build LSTM model
lstm_model = Sequential()

lstm_model.add(
    LSTM(64, input_shape=(30,1))
)

lstm_model.add(Dense(1))

# Compile model
lstm_model.compile(
    optimizer='adam',
    loss='mse'
)

# Train model
lstm_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=32
)

lstm_pred = lstm_model.predict(X_test)

lstm_pred = scaler.inverse_transform(lstm_pred)

y_test_actual = scaler.inverse_transform(
    y_test.reshape(-1, 1)
)

lstm_mae = mean_absolute_error(
    y_test_actual,
    lstm_pred
)

print("LSTM MAE:", lstm_mae)

C:\Users\prana\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 63s 43ms/step - loss: 3.0667e-04
Epoch 2/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 81s 42ms/step - loss: 9.2379e-05
Epoch 3/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 82s 42ms/step - loss: 6.8597e-05
Epoch 4/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 39s 28ms/step - loss: 6.1878e-05
Epoch 5/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 50s 34ms/step - loss: 5.5324e-05
Epoch 6/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 55s 39ms/step - loss: 5.5514e-05
Epoch 7/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 77s 35ms/step - loss: 5.3760e-05
Epoch 8/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 59s 42ms/step - loss: 5.2576e-05
Epoch 9/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 80s 41ms/step - loss: 5.2471e-05
Epoch 10/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 66s 29ms/step - loss: 5.2342e-05
Epoch 11/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 53s 38ms/step - loss: 5.1593e-05
Epoch 12/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 81s 37ms/step - loss: 5.1507e-05
Epoch 13/20
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 77s 33ms/step - loss: 5.1110e-05
Epoch 14/20
1407/1407

Comparing Models

In [106]:
from sklearn.metrics import mean_absolute_error

sarima_mae = mean_absolute_error(
    test['Total'],
    sarima_pred
)

print("SARIMA MAE:", sarima_mae)

SARIMA MAE: 31038055537.098503


Best Model

In [108]:

results = {
    'SARIMA': sarima_mae,
    'Prophet': prophet_mae,
    'XGBoost': xgb_mae,
    'LSTM': lstm_mae
}

print(results)

best_model = min(results, key=results.get)

print("Best Model:", best_model)

{'SARIMA': 31038055537.098503, 'Prophet': 9636909836.567816, 'XGBoost': 34421288.76171876, 'LSTM': 1256787.9240270127}
Best Model: LSTM


In [112]:
import os
import joblib

os.makedirs('models', exist_ok=True)

joblib.dump(
    xgb_model,
    'models/best_model.pkl'
)

print("Model saved successfully.")

Model saved successfully.


In [114]:
import joblib

joblib.dump(
    xgb_model,
    "best_model.pkl"
)

['best_model.pkl']

In [116]:
pip install fastapi uvicorn

In [1]:
import os

print(os.getcwd())

C:\Users\prana
